# Tutorial 2: From Portfolio Optimization to QUBO

This notebook shows how to formulate the Markowitz portfolio selection problem as a
Quadratic Unconstrained Binary Optimization (QUBO) suitable for quantum solvers.

**Key concepts**: QUBO formulation, cardinality constraints, budget penalty tuning.

In [ ]:
import numpy as np
np.random.seed(42)

## 1. The Markowitz QUBO

The binary portfolio selection problem minimizes:

$$\min_x \; \gamma \, x^T \Sigma x - \mu^T x + P \left(\sum_i x_i - K\right)^2$$

where $x \in \{0,1\}^N$, $K$ is the cardinality (number of assets to select),
and $P$ is the budget penalty enforcing exactly $K$ assets.

In [ ]:
# 5-asset problem
mu = np.array([0.12, 0.10, 0.07, 0.03, 0.15])
cov = np.array([
    [0.040, 0.006, 0.002, 0.000, 0.010],
    [0.006, 0.030, 0.004, 0.001, 0.008],
    [0.002, 0.004, 0.020, 0.002, 0.003],
    [0.000, 0.001, 0.002, 0.010, 0.001],
    [0.010, 0.008, 0.003, 0.001, 0.050],
])

print(f"N = {len(mu)} assets")
print(f"Expected returns: {mu}")

## 2. Build the QUBO with qufin

In [ ]:
from qufin.portfolio.qubo import PortfolioQUBO

# Select exactly K=2 assets out of 5, risk aversion gamma=0.5
qubo = PortfolioQUBO(
    mu=mu,
    cov=cov,
    gamma=0.5,
    cardinality=2,
)

print(f"Number of qubits: {qubo.n_qubits}")
print(f"Cardinality: {qubo.cardinality}")
print(f"Encoding: {qubo.encoding}")

## 3. The QUBO Matrix

The QUBO matrix Q encodes the objective. For binary vector x, the cost is x^T Q x.

In [ ]:
Q = qubo.build_matrix()

print(f"QUBO matrix shape: {Q.shape}")
print(f"QUBO matrix:\n{np.round(Q, 4)}")

## 4. Evaluate Candidate Solutions

We can directly evaluate the QUBO cost for any binary assignment.

In [ ]:
from itertools import combinations

n = len(mu)
k = 2

print("All possible 2-asset portfolios:")
best_cost = float("inf")
best_x = None

for combo in combinations(range(n), k):
    x = np.zeros(n)
    x[list(combo)] = 1.0
    cost = x @ Q @ x
    assets = [['Tech','HC','Util','Bond','Ene'][i] for i in combo]
    print(f"  {assets}: cost = {cost:.6f}")
    if cost < best_cost:
        best_cost = cost
        best_x = x.copy()

print(f"\nOptimal selection: {best_x.astype(int)}, cost = {best_cost:.6f}")

## 5. Budget Penalty Tuning

The `budget_penalty` parameter controls how strongly the cardinality constraint is enforced.
Too low: infeasible solutions dominate. Too high: landscape becomes flat.

In [ ]:
for penalty in [0.01, 0.1, 1.0, 10.0]:
    q = PortfolioQUBO(
        mu=mu, cov=cov, gamma=0.5,
        cardinality=2, budget_penalty=penalty,
    )
    Q_p = q.build_matrix()
    # Check energy gap between feasible and infeasible
    feasible = np.array([1, 0, 0, 0, 1])  # 2 assets
    infeasible = np.array([1, 1, 1, 0, 0])  # 3 assets
    e_feas = feasible @ Q_p @ feasible
    e_infeas = infeasible @ Q_p @ infeasible
    print(f"  penalty={penalty:5.2f}  feasible={e_feas:.4f}  infeasible={e_infeas:.4f}  gap={e_infeas - e_feas:.4f}")

## 6. Sector Constraints

qufin supports sector constraints via `sector_map` and `sector_caps`.

In [ ]:
# Assign assets to sectors: 0=Growth, 1=Defensive
sector_map = {0: 0, 1: 0, 4: 0, 2: 1, 3: 1}  # Tech/HC/Ene=Growth, Util/Bond=Defensive
sector_caps = {0: 2, 1: 1}  # Max 2 Growth, 1 Defensive

qubo_sector = PortfolioQUBO(
    mu=mu, cov=cov, gamma=0.5,
    cardinality=3,
    sector_map=sector_map,
    sector_caps=sector_caps,
)

Q_s = qubo_sector.build_matrix()
print(f"Sector-constrained QUBO shape: {Q_s.shape}")

## Summary

In this tutorial we covered:
- The Markowitz QUBO formulation for binary portfolio selection
- Building and inspecting the QUBO matrix with `PortfolioQUBO`
- Exhaustive evaluation of candidate solutions
- Budget penalty tuning for cardinality constraints
- Sector constraints

**Next**: Tutorial 03 solves this QUBO using QAOA on a quantum simulator.